# Vorbereitung

## Importiere Packages
Installiere für dieses Notebook notwendige Packages.

In [5]:
import subprocess

# IFILENAME_REQUIREMENTS = 'requirements.txt'

try:
    from google.colab import drive
    in_colab = True
except ImportError:
    in_colab = False

if in_colab:

    # Install the package 'thefuzz'.
    subprocess.run(['pip', 'install', 'thefuzz'], check=True)

# Ielse:

    # Install all packages listed in local available requirements.txt.
    # Isubprocess.run(['pip', 'install', '-r', FILENAME_REQUIREMENTS], check=True)

- Importiere für dieses Skript weitere notwendige Funktionen.

In [ ]:
import zipfile
import os
from lxml import etree as et
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import matplotlib.pyplot as plt
from pyproj import Transformer
from ipyleaflet import Map, WMSLayer, Circle, Popup
import ipywidgets as widgets
from thefuzz import fuzz, process
import re

 - Packages für geobasierte Visualisierungen

In [ ]:

# 1. Paket im Terminal installieren (nur in Jupyter Notebooks möglich!)
%pip install shapely

# Import packages.
import pandas as pd
from pyproj import Transformer
from ipyleaflet import Map, WMSLayer, Circle, Popup, Heatmap, LayerGroup, LegendControl
import ipywidgets as widgets
import time

# shapely ebenfalls importieren
import shapely

from ipyleaflet import Polygon



Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip3.12 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Definiere Arbeitsverzeichnis für Google Colab
Definiere Arbeitsverzeichnis in Google Colab zur Speicherung Daten und Ergebnisse.

In [ ]:
try:
    from google.colab import drive
    in_colab = True
except ImportError:
    in_colab = False

if in_colab:
  # Define the working directory in google colab.
  WORKING_DIRECTORY = '/content/drive/My Drive/Colab Notebooks/'

  # Mount personal google drive.
  drive.mount('/content/drive')

## Bereite Datengrundlage auf
Lade und entpacke für die Suche verwendete Datengrundlage.

In [ ]:
FILENAME_DATA = 'hgb_corpus_24_07_26_inline_full_updated.zip'
URL_DATA = 'https://github.com/history-unibas/FS2024-Research-Seminar-Demo-Notebook/raw/main/hgb_corpus_24_07_26_inline_full_updated.zip'


if in_colab:

    if not os.path.exists(WORKING_DIRECTORY + FILENAME_DATA):

      # Download the data in working directory.
      subprocess.run(['wget', URL_DATA, '-O', WORKING_DIRECTORY + FILENAME_DATA], check=True)

      # Unzip the file.
      with zipfile.ZipFile(WORKING_DIRECTORY + FILENAME_DATA, 'r') as myzip:
          myzip.extractall(WORKING_DIRECTORY)

    # Update the filepath.
    FILENAME_DATA = WORKING_DIRECTORY + os.path.splitext(FILENAME_DATA)[0] + '.xml'

else:

    # Unzip the file.
    with zipfile.ZipFile(FILENAME_DATA, 'r') as myzip:
        myzip.extractall('.')

    # Update the filepath.
    FILENAME_DATA = os.path.splitext(FILENAME_DATA)[0] + '.xml'

# Element-Orientierte Suche

Im Gegensatz vom früheren Notebook gehen wir grundsätzlich davon aus, dass eure Suche damit beginnt, gewisse XML-Elemente zu suchen. Falls ihr eine Volltextsuche machen wollt, dann modifiziert den XPath einfach so, dass ihr nicht im Body nach Elementen sucht, sondern im Header das Attribut "text" abfragt (siehe alte Demo-Notebooks für Beispiele).

In [ ]:
# Define Search Parameters
YEAR_MIN = 1400
YEAR_MAX = 1530

# Your XPATH Search
# zu erst  wird auf die  Gesamtmenge der  Orgas reduziert, damit wir innerhalb davon weitere Reduktionen (rel) und Filter vornehmen können 


XPATH_EXP = "./Body//Descriptor[@desc_type='tax']/Reference[@entity_type='org']"

#XPATH_EXP = "./Body//Reference[@entity_type = 'org' and @entity_subtype = 'rel']"


In [ ]:
# Load xml file.
context = et.iterparse(FILENAME_DATA, events=('start', 'end'))

# Collector for our results
results = []

# Iterate over each element.
collection_element = None

for event, elem in context:
    if event == 'end' and elem.tag == 'Document':

        # Use XPATH to filter document year
        if not elem.xpath(f"./Header[not(@year < {YEAR_MIN} or @year > {YEAR_MAX})]"):
            continue

        # Add all found xml-elements to our results-list
        results.extend(elem.xpath(XPATH_EXP))
        #results.extend(elem.xpath(XPATH_EXP_2))

print(f"Found {len(results)} elements that matched the xpath criteria!")

# Umwandlung zur Tabelle
Haben wir unsere Elemente gefunden und mögliche Filtern verwendet, können wir die Resultate in ein `Dataframe` umwandeln, quasi eine tabellarische Struktur. Diese kennt ihr schon aus den früheren Demo-Notebooks und könnt dann damit weiterarbeiten.

Denkt daran, dass `result` in diesem Schritt immer noch ein XML-Element beinhaltet, das bedeutet, ihr könnt in eurer Tabelle auch Informationen z.B. über den Parent des Elements verzeichnen.

In [1]:
import re


def clean_whitespaces(text):
    """Replaces all whitespaces with single space characters."""
    return re.sub("\s+", " ", text)


datacollector = []

for result in results:
    data = {}

    # Metadaten zum Dossier
    collection_element = result.xpath("ancestor::Collection")[0]
    data["coll_id"] = collection_element.get("id")
    data["coll_house"] = collection_element.get("house")
    data["coll_x"] = collection_element.get("coord_x")
    data["coll_y"] = collection_element.get("coord_y")

    # Metadaten zum Dokument
    document_element = result.xpath("ancestor::Document")[0]
    data["doc_id"] = document_element.get("id")
    header = document_element.find("Header")
    data["doc_pageNum"] = header.get("pages")
    data["doc_date"] = int(header.get("year"))  # direkt als int
    data["doc_fulltext"] = header.get("text")
    data["doc_image"] = header.get("imagelinks")

    # Suchresultat
    data["res_full"] = clean_whitespaces("".join(result.xpath(".//text()")))
    data["res_head"] = clean_whitespaces("".join(result.find("Head").xpath(".//text()")) if result.find("Head") is not None else "")

    # weiteres
    data["ner_confidence"] = result.get("confidence")

    # Kategorie
    data["res_cat"] = result.get("entity_subtype")

    # Normalisierte Version der Organisation
    data["res_norm"] = result.get("normalization")
    data["res_norm_confidence"] = result.get("norm_confidence")

    datacollector.append(data)

result_df = pd.DataFrame.from_dict(datacollector)

display(result_df)


<>:6: SyntaxWarning: invalid escape sequence '\s'
<>:6: SyntaxWarning: invalid escape sequence '\s'
/var/folders/n4/t9gk6y5969s0ptnyp8q2q_th0000gq/T/ipykernel_18332/1978627693.py:6: SyntaxWarning: invalid escape sequence '\s'
  return re.sub("\s+", " ", text)
/var/folders/n4/t9gk6y5969s0ptnyp8q2q_th0000gq/T/ipykernel_18332/1978627693.py:6: SyntaxWarning: invalid escape sequence '\s'
  return re.sub("\s+", " ", text)


NameError: name 'results' is not defined

In [ ]:
result_df

# Fuzzysuche nach bestimmten Begriffen oder längeren Strings

In [ ]:
pip install rapidfuzz


In [ ]:
from rapidfuzz import fuzz

# bezieht sich auf `result_df`
# wir suchen in der Spalte "doc_fulltext" oder "res_full"

keyword = "Frowen uff Burg"
threshold = 80

# Neue Spalte, die True/False angibt, ob ein Fuzzy-Match über Threshold gefunden wurde
result_df["fuzzy_match"] = result_df["doc_fulltext"].apply(
    lambda text: fuzz.partial_ratio(keyword.lower(), text.lower()) >= threshold if isinstance(text, str) else False
)

# Zeige nur Zeilen mit Match
matched_df = result_df[result_df["fuzzy_match"]]

display(matched_df)

# Institutionsspezifische Suche

#Ausschliesslich diesen String auswechseln mit der gewünschtten Institution

In [ ]:
FILTER_INSTITUTION = "Kloster Klingental"

In [ ]:
klingi_df = result_df[result_df["res_norm"] == FILTER_INSTITUTION]

klingi_df


# Speichern der Tabelle auf dem Desktop

In [ ]:
import os

# Pfad zum Desktop (unabhängig vom Benutzernamen)
desktop_path = os.path.expanduser("~/Desktop/csv")

# HIER KANN DER DATEINAME FESTGELEGT WERDEN
file_name = "klingi.csv"

# Vollständiger Pfad zur Zieldatei
file_path = os.path.join(desktop_path, file_name)

# DataFrame als CSV speichern
klingi_df.to_csv(file_path, index=False)

print(f"Datei wurde gespeichert unter: {file_path}")


## Geovisualisierung

# Transformiere die Koordinaten in WGS 84

In [ ]:
# Transform the coordinates of the search results.
def transform_coords(x, y):
    transformer = Transformer.from_crs(2056, 4326)
    return transformer.transform(x, y)
klingi_df['lat'], klingi_df['lon'] = zip(*klingi_df.apply(
    lambda row: transform_coords(row['coll_x'], row['coll_y']), axis=1
    ))
klingi_df

NameError: name 'klingi_df' is not defined

# Definiere eine Grundkarte


In [ ]:
# Define the basemap.
basemap = WMSLayer(
    url='https://wms.geo.bs.ch/',
    layers='HP_Situationsplan_Basel_1862',
    attribution='Geodaten Kanton Basel-Stadt'
)

# Show the basemap on a map.
m1 = Map(basemap=basemap, center=(47.557, 7.595), zoom=14)
m1

# Institution in Karte einbauen

In [ ]:
# Define a map.
m2 = Map(basemap=basemap, center=(47.557, 7.595), zoom=14)

# Add the search results to the map.
for index, row in klingi_df.iterrows():

    # Create Circle.
    circle = Circle(location=(row['lat'], row['lon']),
                    radius=3,
                    color='red',
                    fill_color='red')

    # Add the circle to the map.
    m2.add(circle)

#  Display the map.
m2

# Polygon definieren: Kleinbasel

In [ ]:


# Add polygon of "Kleinbasel".
polygon_coordinates = [(47.57444952, 7.58560597),
                       (47.57468023, 7.61461808),
                       (47.57468023, 7.61461808),
                       (47.55772285, 7.61381058),
                       (47.55587715, 7.60227495),
                       (47.55619438, 7.59685321),
                       (47.5587899, 7.59154682),
                       (47.56158729, 7.58802845),
                       (47.56663413, 7.58491383),
                       (47.57444952, 7.58560597)]

# Gefilterte Institution in die Kleinbasler Karte einbauen

In [ ]:
from shapely.geometry import Point, Polygon

# Define the polygon
polygon = Polygon(polygon_coordinates)

for index, row in klingi_df.iterrows():

    # Define the point
    point = Point(row['lat'], row['lon'])

    # Check if the point is within the polygon
    klingi_df.at[index, 'in_kleinbasel'] = point.within(polygon)



In [ ]:
klingi_df

Neues DF für KLingi im Kleinbasel

In [ ]:
klingi_in_klein_BS = klingi_df[klingi_df["in_kleinbasel"] == True]

In [ ]:
klingi_in_klein_BS

# Definiere für die Suchresultate ein Pop-up






In [ ]:
# Define a map.
m3 = Map(basemap=basemap, center=(47.557, 7.595), zoom=14)

# Add the search results to the map.
for index, row in klingi_in_klein_BS.iterrows():

    # Create Circle.
    circle = Circle(location=(row['lat'], row['lon']),
                    radius=3,
                    color='blue',
                    fill_color='blue')

    # Add pop-up to display attributes of the circle.
    popup_content = f"""
        ID: {row['coll_id']}<br>
        House: {row['coll_house']}<br>
        Pages: {row['doc_pageNum']}<br>
        Year: {row['doc_date']}<br>
        <a href="{row['doc_image']}">Open Image</a>
        """
    popup = Popup(location=(row['lat'], row['lon']), child=widgets.HTML(popup_content), close_button=True)
    circle.popup = popup

    # Add the circle to the map.
    m3.add(circle)

#  Display the map.
m3

# Suchresultate pro Kategorie
# nur relevant bei top 5

In [ ]:
# Define the column of the dataframe according to which coloring is to take place.
CATEGORY_COLUMN = 'res_cat'

# Define the colors per category.
CATEGORY_COLOR = {'Klingental ': 'red',
                  'St. Alban ': 'blue',
                  'St. Peter ': 'green',
                  'Predigerkloster ': 'orange',
                  'Kirche St. Leonhard': 'pink',
                  'else': 'black'
                  }

# Define a map.
m4 = Map(basemap=basemap, center=(47.557, 7.595), zoom=14)

# Add the search results to the map.
for index, row in geistl_df.iterrows():

    # Determine color based on the value of CATEGORY_COLUMN.
    color = CATEGORY_COLOR.get(row[CATEGORY_COLUMN])
    if color is None:
        color = CATEGORY_COLOR.get('else')

    # Create circle using color based on category.
    circle = Circle(location=(row['lat'], row['lon']),
                    radius=3,
                    color=color,
                    fill_color=color)

    # Add the circle to the map.
    m4.add(circle)

# Add a legend.
legend = LegendControl(legend=CATEGORY_COLOR)
m4.add(legend)

# Display the map.
m4

[Linktext](https://) # Erstelle eine Heatmap

In [ ]:
# Define a map.
m5 = Map(basemap=basemap, center=(47.557, 7.595), zoom=14)

# Create a heatmap layer.
heatmap_layer = Heatmap(locations=st_alban_df[['lat', 'lon']].values.tolist(),
                        radius=5,
                        blur=8,
                        gradient={0.4: 'blue', 0.6: 'cyan', 0.7: 'lime', 0.8: 'yellow', 1.0: 'red'})

# Add the heatmap layer to the map.
m5.add_layer(heatmap_layer)

# Display the map.
m5

# Animierte Visualisierung über die Zeit

In [ ]:
# Time interval in years.
TIME_INTERVAL = 10

# Time in seconds showing each interval.
TIME_STEP = 5

# Define a map.
m6 = Map(basemap=basemap, center=(47.557, 7.595), zoom=14)

#  Display the map.
display(m6)

# Determine minimal and maximal year.
min_year = int(Spitl_df['doc_date'].min())
max_year = int(Spitl_df['doc_date'].max())

# Initialize a group layer.
layer_group = LayerGroup()
m6.add(layer_group)

# Initialize a legend.
legend = LegendControl(legend={})
m6.add(legend)

# Create circles for each time interval.
min_interval = min_year
max_interval = min_interval + TIME_INTERVAL - 1
while min_interval <= max_year:
    # Define the sleeping time.
    time.sleep(TIME_STEP)

    # Define a temporary layer.
    layer_group_new = LayerGroup()

    # Determine the results within the current time interval.
    result_interval = Spitl_df[(Spitl_df['doc_date'] >= min_interval) & (Spitl_df['doc_date'] <= max_interval)]

    for index, row in result_interval.iterrows():
        # Create circle.
        circle = Circle(location=(row['lat'], row['lon']),
                        radius=3,
                        color='blue',
                        fill_color='blue')

        # Add circle to the group layer.
        layer_group_new.add(circle)

    # Update the group layer.
    m6.remove(layer_group)
    layer_group = layer_group_new
    m6.add(layer_group)

    # Update the legend.
    m6.remove(legend)
    legend = LegendControl(legend={f'{min_interval} - {min(max_interval, max_year)}': 'blue'})
    m6.add(legend)

    # Update the interval.
    min_interval += TIME_INTERVAL
    max_interval += TIME_INTERVAL

# Interval 33 Years

In [ ]:
# Time interval in years.
TIME_INTERVAL = 33

# Time in seconds showing each interval.
TIME_STEP = 3

# Define a map.
m6 = Map(basemap=basemap, center=(47.557, 7.595), zoom=14)

#  Display the map.
display(m6)

# Determine minimal and maximal year.
min_year = int(Spitl_df['doc_date'].min())
max_year = int(Spitl_df['doc_date'].max())

# Initialize a group layer.
layer_group = LayerGroup()
m6.add(layer_group)

# Initialize a legend.
legend = LegendControl(legend={})
m6.add(legend)

# Create circles for each time interval.
min_interval = min_year
max_interval = min_interval + TIME_INTERVAL - 1
while min_interval <= max_year:
    # Define the sleeping time.
    time.sleep(TIME_STEP)

    # Define a temporary layer.
    layer_group_new = LayerGroup()

    # Determine the results within the current time interval.
    result_interval = Spitl_df[(Spitl_df['doc_date'] >= min_interval) & (Spitl_df['doc_date'] <= max_interval)]

    for index, row in result_interval.iterrows():
        # Create circle.
        circle = Circle(location=(row['lat'], row['lon']),
                        radius=3,
                        color='blue',
                        fill_color='blue')

        # Add circle to the group layer.
        layer_group_new.add(circle)

    # Update the group layer.
    m6.remove(layer_group)
    layer_group = layer_group_new
    m6.add(layer_group)

    # Update the legend.
    m6.remove(legend)
    legend = LegendControl(legend={f'{min_interval} - {min(max_interval, max_year)}': 'blue'})
    m6.add(legend)

    # Update the interval.
    min_interval += TIME_INTERVAL
    max_interval += TIME_INTERVAL